In [1]:
# Cell 1 — check we're using the project's virtual environment
import sys
from pathlib import Path
import pandas as pd
import matplotlib

print("Python:", sys.version.split()[0])
print("Running from:", sys.executable)
print("Inside .venv?", ".venv" in sys.executable)
print("pandas:", pd.__version__, "| matplotlib:", matplotlib.__version__)

Python: 3.13.15
Running from: /Users/brandonrichardson/Documents/GitHub/fitness-data-analyst/.venv/bin/python
Inside .venv? True
pandas: 3.0.6 | matplotlib: 3.11.2


In [2]:
# Cell 2 — locate the project folder and list every CSV file
PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT / "data" / "raw"
csv_files = sorted(RAW.rglob("*.csv"))   # rglob = search all sub-folders

print("Project folder:", PROJECT)
print("CSV files found:", len(csv_files))
for f in csv_files:
    print(f"{f.stat().st_size / 1_000_000:7.2f} MB  {f.name}")

Project folder: /Users/brandonrichardson/Documents/GitHub/fitness-data-analyst
CSV files found: 18
   0.11 MB  dailyActivity_merged.csv
   0.03 MB  dailyCalories_merged.csv
   0.07 MB  dailyIntensities_merged.csv
   0.03 MB  dailySteps_merged.csv
  89.59 MB  heartrate_seconds_merged.csv
   0.80 MB  hourlyCalories_merged.csv
   0.90 MB  hourlyIntensities_merged.csv
   0.80 MB  hourlySteps_merged.csv
  66.44 MB  minuteCaloriesNarrow_merged.csv
  22.99 MB  minuteCaloriesWide_merged.csv
  46.36 MB  minuteIntensitiesNarrow_merged.csv
   3.31 MB  minuteIntensitiesWide_merged.csv
  47.69 MB  minuteMETsNarrow_merged.csv
   8.85 MB  minuteSleep_merged.csv
  46.53 MB  minuteStepsNarrow_merged.csv
   3.48 MB  minuteStepsWide_merged.csv
   0.02 MB  sleepDay_merged.csv
   0.01 MB  weightLogInfo_merged.csv


In [3]:
# Cell 3 — load the four tables we'll care about most
def load(name):
    matches = [f for f in csv_files if f.name == name]
    assert len(matches) == 1, f"Expected 1 file called {name}, found {len(matches)}"
    return pd.read_csv(matches[0])

daily  = load("dailyActivity_merged.csv")
sleep  = load("sleepDay_merged.csv")
weight = load("weightLogInfo_merged.csv")
heart  = load("heartrate_seconds_merged.csv")   # ~2.5 million rows: takes a few seconds

tables = {"daily": daily, "sleep": sleep, "weight": weight, "heart": heart}
for name, df in tables.items():
    print(f"{name:7s} rows={len(df):>10,}  users={df['Id'].nunique():>3}  columns={df.shape[1]}")
    print("         ", list(df.columns))

daily   rows=       940  users= 33  columns=15
          ['Id', 'ActivityDate', 'TotalSteps', 'TotalDistance', 'TrackerDistance', 'LoggedActivitiesDistance', 'VeryActiveDistance', 'ModeratelyActiveDistance', 'LightActiveDistance', 'SedentaryActiveDistance', 'VeryActiveMinutes', 'FairlyActiveMinutes', 'LightlyActiveMinutes', 'SedentaryMinutes', 'Calories']
sleep   rows=       413  users= 24  columns=5
          ['Id', 'SleepDay', 'TotalSleepRecords', 'TotalMinutesAsleep', 'TotalTimeInBed']
weight  rows=        67  users=  8  columns=8
          ['Id', 'Date', 'WeightKg', 'WeightPounds', 'Fat', 'BMI', 'IsManualReport', 'LogId']
heart   rows= 2,483,658  users= 14  columns=3
          ['Id', 'Time', 'Value']


In [4]:
# Cell 4 — parse dates into NEW columns (originals stay untouched)
daily["activity_date"] = pd.to_datetime(daily["ActivityDate"], format="%m/%d/%Y")
sleep["sleep_date"]    = pd.to_datetime(sleep["SleepDay"],     format="%m/%d/%Y %I:%M:%S %p")
weight["log_time"]     = pd.to_datetime(weight["Date"],        format="%m/%d/%Y %I:%M:%S %p")
heart["reading_time"]  = pd.to_datetime(heart["Time"],         format="%m/%d/%Y %I:%M:%S %p")

for name, df, col in [("daily", daily, "activity_date"), ("sleep", sleep, "sleep_date"),
                      ("weight", weight, "log_time"), ("heart", heart, "reading_time")]:
    print(f"{name:7s} {df[col].min()}  →  {df[col].max()}")

daily   2016-04-12 00:00:00  →  2016-05-12 00:00:00
sleep   2016-04-12 00:00:00  →  2016-05-12 00:00:00
weight  2016-04-12 06:47:11  →  2016-05-12 23:59:59
heart   2016-04-12 00:00:00  →  2016-05-12 16:20:00


In [5]:
# Cell 5 — sanity checks, then prove the checks can catch a known-bad case
def sanity_checks(daily, sleep, heart):
    first, last = pd.Timestamp("2016-04-12"), pd.Timestamp("2016-05-12")
    minute_cols = ["VeryActiveMinutes", "FairlyActiveMinutes",
                   "LightlyActiveMinutes", "SedentaryMinutes"]
    return [
        ("Daily dates inside the export window (12 Apr – 12 May 2016)",
         daily["activity_date"].between(first, last).all()),
        ("Steps are never negative",
         (daily["TotalSteps"] >= 0).all()),
        ("No day has more than 1,440 minutes (24 h) recorded",
         (daily[minute_cols].sum(axis=1) <= 1440).all()),
        ("Minutes asleep never exceed minutes in bed",
         (sleep["TotalMinutesAsleep"] <= sleep["TotalTimeInBed"]).all()),
        ("Heart rate always between 30 and 220 bpm",
         heart["Value"].between(30, 220).all()),
        ("At most one daily-activity row per user per date",
         not daily.duplicated(["Id", "activity_date"]).any()),
        ("No exact duplicate rows in sleep",
         not sleep.duplicated().any()),
        ("Every sleep user also appears in daily activity",
         set(sleep["Id"]) <= set(daily["Id"])),
    ]

print("=== Real data ===")
for desc, ok in sanity_checks(daily, sleep, heart):
    print(("PASS  " if ok else "FAIL  ") + desc)

# Known-bad test: break two things on purpose. These two checks MUST say FAIL.
bad_daily = daily.copy()
bad_daily.loc[bad_daily.index[0], "TotalSteps"] = -100
bad_sleep = sleep.copy()
bad_sleep.loc[bad_sleep.index[0], "TotalMinutesAsleep"] = bad_sleep["TotalTimeInBed"].iloc[0] + 10

print("\n=== Deliberately broken copy (steps + sleep checks should FAIL) ===")
for desc, ok in sanity_checks(bad_daily, bad_sleep, heart):
    print(("PASS  " if ok else "FAIL  ") + desc)

=== Real data ===
PASS  Daily dates inside the export window (12 Apr – 12 May 2016)
PASS  Steps are never negative
PASS  No day has more than 1,440 minutes (24 h) recorded
PASS  Minutes asleep never exceed minutes in bed
PASS  Heart rate always between 30 and 220 bpm
PASS  At most one daily-activity row per user per date
FAIL  No exact duplicate rows in sleep
PASS  Every sleep user also appears in daily activity

=== Deliberately broken copy (steps + sleep checks should FAIL) ===
PASS  Daily dates inside the export window (12 Apr – 12 May 2016)
FAIL  Steps are never negative
PASS  No day has more than 1,440 minutes (24 h) recorded
FAIL  Minutes asleep never exceed minutes in bed
PASS  Heart rate always between 30 and 220 bpm
PASS  At most one daily-activity row per user per date
FAIL  No exact duplicate rows in sleep
PASS  Every sleep user also appears in daily activity


In [6]:
# Cell 6 — compare with figures widely reported for this export, and with Zenodo's description
expected = {"daily": (940, 33), "sleep": (413, 24), "weight": (67, 8), "heart": (2_483_658, 14)}
for name, (exp_rows, exp_users) in expected.items():
    df = tables[name]
    rows_ok  = "✓" if len(df) == exp_rows else "✗"
    users_ok = "✓" if df["Id"].nunique() == exp_users else "✗"
    print(f"{name:7s} rows {len(df):>10,} (expected {exp_rows:>10,}) {rows_ok}   "
          f"users {df['Id'].nunique():>3} (expected {exp_users}) {users_ok}")

print("\nZenodo's description says 30 participants. Distinct users in daily activity:",
      daily["Id"].nunique())

daily   rows        940 (expected        940) ✓   users  33 (expected 33) ✓
sleep   rows        413 (expected        413) ✓   users  24 (expected 24) ✓
weight  rows         67 (expected         67) ✓   users   8 (expected 8) ✓
heart   rows  2,483,658 (expected  2,483,658) ✓   users  14 (expected 14) ✓

Zenodo's description says 30 participants. Distinct users in daily activity: 33


In [7]:
# Cell 7 — eyeball a few rows of each table
for name, df in tables.items():
    print(f"\n--- {name} ---")
    display(df.head(3))


--- daily ---


,Id,ActivityDate,TotalSteps,TotalDistance,TrackerDistance,LoggedActivitiesDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,activity_date
0,1503960366,4/12/2016,13162,8.50,8.50,0.0,1.88,0.55,6.06,0.0,25,13,328,728,1985,2016-04-12
1,1503960366,4/13/2016,10735,6.97,6.97,0.0,1.57,0.69,4.71,0.0,21,19,217,776,1797,2016-04-13
2,1503960366,4/14/2016,10460,6.74,6.74,0.0,2.44,0.40,3.91,0.0,30,11,181,1218,1776,2016-04-14



--- sleep ---


,Id,SleepDay,TotalSleepRecords,TotalMinutesAsleep,TotalTimeInBed,sleep_date
0,1503960366,4/12/2016 12:00:00 AM,1,327,346,2016-04-12
1,1503960366,4/13/2016 12:00:00 AM,2,384,407,2016-04-13
2,1503960366,4/15/2016 12:00:00 AM,1,412,442,2016-04-15



--- weight ---


,Id,Date,WeightKg,WeightPounds,Fat,BMI,IsManualReport,LogId,log_time
0,1503960366,5/2/2016 11:59:59 PM,52.599998,115.963147,22.0,22.650000,True,1462233599000,2016-05-02 23:59:59
1,1503960366,5/3/2016 11:59:59 PM,52.599998,115.963147,NaN,22.650000,True,1462319999000,2016-05-03 23:59:59
2,1927972279,4/13/2016 1:08:52 AM,133.500000,294.317120,NaN,47.540001,False,1460509732000,2016-04-13 01:08:52



--- heart ---


,Id,Time,Value,reading_time
0,2022484408,4/12/2016 7:21:00 AM,97,2016-04-12 07:21:00
1,2022484408,4/12/2016 7:21:05 AM,102,2016-04-12 07:21:05
2,2022484408,4/12/2016 7:21:10 AM,105,2016-04-12 07:21:10


In [8]:
# Cell 8 — investigate the duplicate sleep rows (look, don't change anything yet)
dupes = sleep[sleep.duplicated(keep=False)]          # keep=False shows BOTH copies
print("Rows that are exact copies of another row:", sleep.duplicated().sum())
print("Users affected:", dupes["Id"].nunique())
print("Sleep rows if duplicates removed:", len(sleep) - sleep.duplicated().sum(), "of", len(sleep))
display(dupes.sort_values(["Id", "sleep_date"]))

Rows that are exact copies of another row: 3
Users affected: 3
Sleep rows if duplicates removed: 410 of 413


,Id,SleepDay,TotalSleepRecords,TotalMinutesAsleep,TotalTimeInBed,sleep_date
160,4388161847,5/5/2016 12:00:00 AM,1,471,495,2016-05-05
161,4388161847,5/5/2016 12:00:00 AM,1,471,495,2016-05-05
222,4702921684,5/7/2016 12:00:00 AM,1,520,543,2016-05-07
223,4702921684,5/7/2016 12:00:00 AM,1,520,543,2016-05-07
379,8378563200,4/25/2016 12:00:00 AM,1,388,402,2016-04-25
380,8378563200,4/25/2016 12:00:00 AM,1,388,402,2016-04-25
